# Comparación de columnas — 2020 a 2024

Objetivo: identificar las columnas que cambian entre años para los tres datasets
(nacimientos, fetales, no fetales) antes de construir el pipeline de unificación.

Ejecutar este notebook **antes** de `procesamiento2020.ipynb` y los demás años.

In [4]:
import os
import sys
import pandas as pd

sys.path.append(os.path.abspath('..'))

from etl.extract import load_csv
from etl.config import YEARS, DATASETS



In [5]:
from pathlib import Path

RAW_DATA = Path('../data/raw')
YEARS = [2020, 2021, 2022, 2023, 2024]

# Prefijos de archivo tal como están en data/raw/
PREFIJOS = {
    'nacimientos': 'nac',
    'fetales':     'fetal',
    'no_fetales':  'nofetal'
}

def leer_columnas(prefijo: str, año: int) -> list:
    """Lee solo las columnas (nrows=0) de un archivo CSV/XLS."""
    for ext in ['.csv', '.CSV', '.xlsx', '.xls']:
        path = RAW_DATA / f"{prefijo}{año}{ext}"
        if path.exists():
            try:
                if ext in ['.xlsx', '.xls']:
                    df = pd.read_excel(path, nrows=0)
                else:
                    df = pd.read_csv(path, nrows=0, sep=',',
                                     encoding='latin-1', low_memory=False)
                return list(df.columns)
            except Exception as e:
                print(f"  Error leyendo {path.name}: {e}")
    print(f"  Archivo no encontrado: {prefijo}{año}.*")
    return []

# Construir diccionario {dataset: {año: [columnas]}}
columnas = {}
for dataset, prefijo in PREFIJOS.items():
    columnas[dataset] = {}
    print(f"\n{'='*50}\n  {dataset.upper()}\n{'='*50}")
    for año in YEARS:
        cols = leer_columnas(prefijo, año)
        columnas[dataset][año] = cols
        print(f"  {año}: {len(cols)} columnas")

print("\n✓ Carga de headers completada.")


  NACIMIENTOS
  2020: 37 columnas
  2021: 39 columnas
  2022: 39 columnas
  2023: 39 columnas
  2024: 40 columnas

  FETALES
  2020: 43 columnas
  2021: 44 columnas
  2022: 44 columnas
  2023: 46 columnas
  2024: 46 columnas

  NO_FETALES
  2020: 55 columnas
  2021: 56 columnas
  2022: 57 columnas
  2023: 59 columnas
  2024: 61 columnas

✓ Carga de headers completada.


## 2. Función de diagnóstico

In [ ]:
def diagnostico_columnas(cols_por_año: dict, nombre_dataset: str):
    """
    Genera tres salidas:
    1. Matriz de presencia/ausencia (✓ / ✗)
    2. Columnas que están en TODOS los años (columnas estables)
    3. Columnas que faltan en al menos un año (columnas inconsistentes)
    """
    años = [a for a in cols_por_año if cols_por_año[a]] # list comprehension (no vacios).
    todas = sorted(set(c for cols in cols_por_año.values() for c in cols)) # lista única de todas las columnas que han existido en cualquier año.

    # Matriz de presencia
    matriz = pd.DataFrame(
        {año: ['✓' if col in cols_por_año[año] else '✗' for col in todas]
         for año in años},
        index=todas
    )

    # Columnas estables (presentes en todos los años disponibles)
    estables = [col for col in todas
                if all(col in cols_por_año[a] for a in años)]

    # Columnas inconsistentes, columnas que aparecen en algunos años pero no en todos.
    inconsistentes = matriz[matriz.apply(lambda r: '✗' in r.values, axis=1)]

    print(f"\n{'='*60}")
    print(f"  DATASET: {nombre_dataset.upper()}")
    print(f"{'='*60}")
    print(f"  Total columnas únicas entre todos los años : {len(todas)}")
    print(f"  Columnas estables (en todos los años)      : {len(estables)}")
    print(f"  Columnas inconsistentes (faltan en algún año): {len(inconsistentes)}")

    print(f"\n--- Matriz de presencia ---")
    display(matriz)

    if len(inconsistentes) > 0:
        print(f"\n--- Columnas inconsistentes ---")
        display(inconsistentes)

    print(f"\n--- Columnas estables ({len(estables)}) ---")
    print(estables)

    return matriz, estables, inconsistentes

### 3. Diagnóstico — Nacimientos

In [7]:
mat_nac, estables_nac, incons_nac = diagnostico_columnas(
    columnas['nacimientos'], 'Nacimientos'
)


  DATASET: NACIMIENTOS
  Total columnas únicas entre todos los años : 40
  Columnas estables (en todos los años)      : 37
  Columnas inconsistentes (faltan en algún año): 3

--- Matriz de presencia ---


,2020,2021,2022,2023,2024
ANO,✓,✓,✓,✓,✓
APGAR1,✓,✓,✓,✓,✓
APGAR2,✓,✓,✓,✓,✓
AREANAC,✓,✓,✓,✓,✓
AREA_RES,✓,✓,✓,✓,✓
ATEN_PAR,✓,✓,✓,✓,✓
CODMUNRE,✓,✓,✓,✓,✓
CODPAISNACMAD,✗,✗,✗,✗,✓
CODPRES,✓,✓,✓,✓,✓
CODPTORE,✓,✓,✓,✓,✓



--- Columnas inconsistentes ---


,2020,2021,2022,2023,2024
CODPAISNACMAD,✗,✗,✗,✗,✓
TIPOFORMULARIO,✗,✓,✓,✓,✓
T_GES_AGRU_CIE,✗,✓,✓,✓,✓



--- Columnas estables (37) ---
['ANO', 'APGAR1', 'APGAR2', 'AREANAC', 'AREA_RES', 'ATEN_PAR', 'CODMUNRE', 'CODPRES', 'CODPTORE', 'COD_DPTO', 'COD_MUNIC', 'EDAD_MADRE', 'EDAD_PADRE', 'EST_CIVM', 'FECHA_NACM', 'IDCLASADMI', 'IDFACTORRH', 'IDHEMOCLAS', 'IDPERTET', 'MES', 'MUL_PARTO', 'NIV_EDUM', 'NIV_EDUP', 'NUMCONSUL', 'N_EMB', 'N_HIJOSV', 'OTRO_SIT', 'PESO_NAC', 'PROFESION', 'SEG_SOCIAL', 'SEXO', 'SIT_PARTO', 'TALLA_NAC', 'TIPO_PARTO', 'T_GES', 'ULTCURMAD', 'ULTCURPAD']


## 4. Diagnóstico - Fetales

In [13]:
mat_fet, estables_fet, incons_fet = diagnostico_columnas(
    columnas['fetales'], 'Fetales'
)


  DATASET: FETALES
  Total columnas únicas entre todos los años : 48
  Columnas estables (en todos los años)      : 41
  Columnas inconsistentes (faltan en algún año): 7

--- Matriz de presencia ---


,2020,2021,2022,2023,2024
ANO,✓,✓,✓,✓,✓
AREA_RES,✓,✓,✓,✓,✓
ASIS_MED,✓,✓,✓,✓,✓
A_DEFUN,✓,✓,✓,✓,✓
CAUSA_667,✓,✓,✓,✓,✓
CAUSA_MULT,✓,✓,✓,✓,✓
CAU_HOMOL,✓,✓,✓,✓,✓
CODMUNOC,✓,✓,✓,✓,✓
CODMUNRE,✓,✓,✓,✓,✓
CODOCUR,✓,✓,✓,✓,✓



--- Columnas inconsistentes ---


,2020,2021,2022,2023,2024
CODPAISNACMAD,✗,✗,✗,✗,✓
C_MUERTEF,✗,✗,✗,✓,✓
C_MUERTEG,✗,✗,✗,✓,✓
IDADMISALUD,✓,✓,✓,✓,✗
REGSOCIALMADRE,✗,✗,✗,✗,✓
SEG_SOCIAL,✓,✓,✓,✓,✗
T_GES_AGRU_CIE,✗,✓,✓,✓,✓



--- Columnas estables (41) ---
['ANO', 'AREA_RES', 'ASIS_MED', 'A_DEFUN', 'CAUSA_667', 'CAUSA_MULT', 'CAU_HOMOL', 'CODMUNOC', 'CODMUNRE', 'CODOCUR', 'CODPRES', 'CODPTORE', 'COD_DPTO', 'COD_MUNIC', 'CONS_EXP', 'C_BAS1', 'C_MUERTE', 'C_MUERTEB', 'C_MUERTEC', 'C_MUERTED', 'C_MUERTEE', 'EDAD_MADRE', 'EST_CIVM', 'HORA', 'IDPROFCER', 'MES', 'MINUTOS', 'MU_PARTO', 'NIV_EDUM', 'N_HIJOSM', 'N_HIJOSV', 'OTRSITIODE', 'PESO_NAC', 'P_PMAN_IRIS', 'SEXO', 'SIT_DEFUN', 'TIPO_DEFUN', 'TIPO_EMB', 'T_GES', 'T_PARTO', 'ULTCURMAD']


## 5. Diagnóstico - No Fetales

In [8]:
mat_nof, estables_nof, incons_nof = diagnostico_columnas(
    columnas['no_fetales'], 'No Fetales'
)


  DATASET: NO FETALES
  Total columnas únicas entre todos los años : 62
  Columnas estables (en todos los años)      : 54
  Columnas inconsistentes (faltan en algún año): 8

--- Matriz de presencia ---


,2020,2021,2022,2023,2024
ANO,✓,✓,✓,✓,✓
AREA_RES,✓,✓,✓,✓,✓
ASIS_MED,✓,✓,✓,✓,✓
A_DEFUN,✓,✓,✓,✓,✓
CAUSA_667,✓,✓,✓,✓,✓
...,...,...,...,...,...
T_GES,✓,✓,✓,✓,✓
T_GES_AGRU_CIE,✗,✓,✓,✓,✓
T_PARTO,✓,✓,✓,✓,✓
ULTCURFAL,✓,✓,✓,✓,✓



--- Columnas inconsistentes ---


,2020,2021,2022,2023,2024
CODPAISNACFAL,✗,✗,✗,✗,✓
CODPAISNACMAD,✗,✗,✗,✗,✓
C_MUERTEF,✗,✗,✗,✓,✓
C_MUERTEG,✗,✗,✗,✓,✓
IDADMISALUD,✓,✓,✓,✓,✗
REGSOCIALMADRE,✗,✗,✗,✗,✓
TIPOFORMULARIO,✗,✗,✓,✓,✓
T_GES_AGRU_CIE,✗,✓,✓,✓,✓



--- Columnas estables (54) ---
['ANO', 'AREA_RES', 'ASIS_MED', 'A_DEFUN', 'CAUSA_667', 'CAUSA_MULT', 'CAU_HOMOL', 'CODMUNOC', 'CODMUNRE', 'CODOCUR', 'CODPRES', 'CODPTORE', 'COD_DPTO', 'COD_MUNIC', 'CONS_EXP', 'C_BAS1', 'C_MUERTE', 'C_MUERTEB', 'C_MUERTEC', 'C_MUERTED', 'C_MUERTEE', 'EDAD_MADRE', 'EMB_FAL', 'EMB_MES', 'EMB_SEM', 'EST_CIVIL', 'EST_CIVM', 'GRU_ED1', 'GRU_ED2', 'HORA', 'IDPERTET', 'IDPROFCER', 'MES', 'MINUTOS', 'MUERTEPORO', 'MU_PARTO', 'NIVEL_EDU', 'NIV_EDUM', 'N_HIJOSM', 'N_HIJOSV', 'OCUPACION', 'OTRSITIODE', 'PESO_NAC', 'P_PMAN_IRIS', 'SEG_SOCIAL', 'SEXO', 'SIMUERTEPO', 'SIT_DEFUN', 'TIPO_DEFUN', 'TIPO_EMB', 'T_GES', 'T_PARTO', 'ULTCURFAL', 'ULTCURMAD']


## 6. Columnas que se eliminan según colEliminar.md

Verificamos que todas las columnas a borrar existen realmente
en los archivos de cada año.>

In [11]:
import re

def leer_cols_eliminar(ruta_md: str) -> dict:
    """Parsea colEliminar.md y retorna {seccion: [columnas]}."""
    resultado = {}
    with open(ruta_md, encoding='utf-8') as f:
        contenido = f.read()
    secciones = re.split(r'##\s+', contenido)[1:]  # quitar primer bloque vacío
    for seccion in secciones:
        lineas = seccion.strip().splitlines()
        nombre = lineas[0].strip()
        cols = [l.strip() for l in lineas[1:] if l.strip() and not l.startswith('#')]
        resultado[nombre] = cols
    return resultado

cols_eliminar = leer_cols_eliminar('../colEliminar.md')

mapeo_seccion_dataset = {
    'Nacimientos': ('nacimientos', cols_eliminar.get('Nacimientos', [])),
    'Fetales':     ('fetales',     cols_eliminar.get('Fetales', [])),
    'No fetales':  ('no_fetales',  cols_eliminar.get('No fetales', [])),
}

print("Verificación de columnas a eliminar por año:\n")
for seccion, (dataset, cols_borrar) in mapeo_seccion_dataset.items():
    print(f"{'─'*50}")
    print(f"  {seccion} — {len(cols_borrar)} columnas declaradas para eliminar")
    print(f"{'─'*50}")
    for año in YEARS:
        cols_año = columnas[dataset].get(año, [])
        presentes  = [c for c in cols_borrar if c in cols_año]
        ausentes   = [c for c in cols_borrar if c not in cols_año]
        print(f"  {año}: {len(presentes)} presentes", end='')
        if ausentes:
            print(f"  | ⚠ No encontradas: {ausentes}")
        else:
            print(" ✓")

Verificación de columnas a eliminar por año:

──────────────────────────────────────────────────
  Nacimientos — 9 columnas declaradas para eliminar
──────────────────────────────────────────────────
  2020: 9 presentes ✓
  2021: 9 presentes ✓
  2022: 9 presentes ✓
  2023: 9 presentes ✓
  2024: 9 presentes ✓
──────────────────────────────────────────────────
  Fetales — 18 columnas declaradas para eliminar
──────────────────────────────────────────────────
  2020: 18 presentes ✓
  2021: 18 presentes ✓
  2022: 18 presentes ✓
  2023: 18 presentes ✓
  2024: 18 presentes ✓
──────────────────────────────────────────────────
  No fetales — 20 columnas declaradas para eliminar
──────────────────────────────────────────────────
  2020: 20 presentes ✓
  2021: 20 presentes ✓
  2022: 20 presentes ✓
  2023: 20 presentes ✓
  2024: 20 presentes ✓


## 7. Resumen final — columnas candidatas para el schema

Lista definitiva de columnas que **quedarán** después de eliminar
las declaradas en `colEliminar.md`, cruzada con las estables entre años.
Esta lista es el insumo para completar `etl/schema.py`.

In [15]:
def columnas_finales(estables: list, cols_borrar: list, nombre: str):
    """Columnas estables que NO están en la lista de eliminación."""
    # Comparar en mayúsculas porque los archivos vienen en mayúsculas
    borrar_set = set(c.upper() for c in cols_borrar)
    finales = [c for c in estables if c.upper() not in borrar_set]
    print(f"\n{'='*55}")
    print(f"  {nombre} — columnas finales para análisis: {len(finales)}")
    print(f"{'='*55}")
    for c in finales:
        print(f"    {c}")
    return finales

finales_nac = columnas_finales(
    estables_nac,
    cols_eliminar.get('Nacimientos', []),
    'NACIMIENTOS'
)




  NACIMIENTOS — columnas finales para análisis: 28
    ANO
    AREANAC
    AREA_RES
    ATEN_PAR
    CODMUNRE
    CODPRES
    CODPTORE
    COD_DPTO
    COD_MUNIC
    EDAD_MADRE
    EDAD_PADRE
    EST_CIVM
    IDCLASADMI
    IDFACTORRH
    IDHEMOCLAS
    MES
    MUL_PARTO
    NIV_EDUM
    NIV_EDUP
    NUMCONSUL
    PESO_NAC
    PROFESION
    SEG_SOCIAL
    SEXO
    SIT_PARTO
    TALLA_NAC
    TIPO_PARTO
    T_GES


In [16]:
finales_fet = columnas_finales(
    estables_fet,
    cols_eliminar.get('Fetales', []),
    'FETALES'
)




  FETALES — columnas finales para análisis: 23
    ANO
    AREA_RES
    ASIS_MED
    A_DEFUN
    CAU_HOMOL
    CODMUNRE
    CODPTORE
    COD_DPTO
    COD_MUNIC
    EDAD_MADRE
    EST_CIVM
    MES
    MU_PARTO
    NIV_EDUM
    N_HIJOSM
    N_HIJOSV
    PESO_NAC
    SEXO
    SIT_DEFUN
    TIPO_DEFUN
    TIPO_EMB
    T_GES
    T_PARTO


In [17]:
finales_nof = columnas_finales(
    estables_nof,
    cols_eliminar.get('No fetales', []),
    'NO FETALES'
)


  NO FETALES — columnas finales para análisis: 34
    ANO
    AREA_RES
    ASIS_MED
    A_DEFUN
    CODMUNRE
    CODPTORE
    COD_DPTO
    COD_MUNIC
    EDAD_MADRE
    EMB_FAL
    EMB_MES
    EMB_SEM
    EST_CIVIL
    EST_CIVM
    GRU_ED1
    GRU_ED2
    IDPERTET
    MES
    MUERTEPORO
    NIVEL_EDU
    NIV_EDUM
    N_HIJOSM
    N_HIJOSV
    OCUPACION
    PESO_NAC
    SEG_SOCIAL
    SEXO
    SIMUERTEPO
    SIT_DEFUN
    TIPO_DEFUN
    TIPO_EMB
    T_GES
    T_PARTO
    ULTCURFAL


## Borrardor para el schema ETL <


In [24]:
import re

def a_nombre_estandar(col: str) -> str:
    """Aplica la misma lógica de etl/normalization.py."""
    c = col.strip().lower()
    c = re.sub(r'\s+', '_', c)
    for viejo, nuevo in [('á','a'),('é','e'),('í','i'),('ó','o'),('ú','u')]:
        c = c.replace(viejo, nuevo)
    c = re.sub(r'[^a-z0-9_]', '', c)
    return c

def generar_schema(finales: list, nombre_var: str):
    cols_std = sorted(set(a_nombre_estandar(c) for c in finales))

    contenido = "\n".join([f'    "{c}",' for c in cols_std])

    output = f"{nombre_var} = [\n{contenido}\n]"

    print(output)
    return output

print("# Contenido para etl/schema.py\n")
print("# Columnas válidas tras ETL (estandarizadas a minúsculas)\n")
generar_schema(finales_nac, 'VALID_COLUMNS_NACIMIENTOS') 


# Contenido para etl/schema.py

# Columnas válidas tras ETL (estandarizadas a minúsculas)

VALID_COLUMNS_NACIMIENTOS = [
    "ano",
    "area_res",
    "areanac",
    "aten_par",
    "cod_dpto",
    "cod_munic",
    "codmunre",
    "codpres",
    "codptore",
    "edad_madre",
    "edad_padre",
    "est_civm",
    "idclasadmi",
    "idfactorrh",
    "idhemoclas",
    "mes",
    "mul_parto",
    "niv_edum",
    "niv_edup",
    "numconsul",
    "peso_nac",
    "profesion",
    "seg_social",
    "sexo",
    "sit_parto",
    "t_ges",
    "talla_nac",
    "tipo_parto",
]


'VALID_COLUMNS_NACIMIENTOS = [\n    "ano",\n    "area_res",\n    "areanac",\n    "aten_par",\n    "cod_dpto",\n    "cod_munic",\n    "codmunre",\n    "codpres",\n    "codptore",\n    "edad_madre",\n    "edad_padre",\n    "est_civm",\n    "idclasadmi",\n    "idfactorrh",\n    "idhemoclas",\n    "mes",\n    "mul_parto",\n    "niv_edum",\n    "niv_edup",\n    "numconsul",\n    "peso_nac",\n    "profesion",\n    "seg_social",\n    "sexo",\n    "sit_parto",\n    "t_ges",\n    "talla_nac",\n    "tipo_parto",\n]'

In [19]:
generar_schema(finales_fet, 'VALID_COLUMNS_FETALES')


VALID_COLUMNS_FETALES = [
    "a_defun",
    "ano",
    "area_res",
    "asis_med",
    "cau_homol",
    "cod_dpto",
    "cod_munic",
    "codmunre",
    "codptore",
    "edad_madre",
    "est_civm",
    "mes",
    "mu_parto",
    "n_hijosm",
    "n_hijosv",
    "niv_edum",
    "peso_nac",
    "sexo",
    "sit_defun",
    "t_ges",
    "t_parto",
    "tipo_defun",
    "tipo_emb",
]



In [20]:
generar_schema(finales_nof, 'VALID_COLUMNS_NO_FETALES')

VALID_COLUMNS_NO_FETALES = [
    "a_defun",
    "ano",
    "area_res",
    "asis_med",
    "cod_dpto",
    "cod_munic",
    "codmunre",
    "codptore",
    "edad_madre",
    "emb_fal",
    "emb_mes",
    "emb_sem",
    "est_civil",
    "est_civm",
    "gru_ed1",
    "gru_ed2",
    "idpertet",
    "mes",
    "muerteporo",
    "n_hijosm",
    "n_hijosv",
    "niv_edum",
    "nivel_edu",
    "ocupacion",
    "peso_nac",
    "seg_social",
    "sexo",
    "simuertepo",
    "sit_defun",
    "t_ges",
    "t_parto",
    "tipo_defun",
    "tipo_emb",
    "ultcurfal",
]

